# Appendix A4: Deep Active Inference on Cognitive Tasks
## Building Generative Models for Go/No-Go, Evidence Accumulation, and Working Memory

**Spinning Up in Active Inference** | Appendix

---

Active Inference has been demonstrated on grid worlds, T-mazes, and toy POMDPs. But cognitive neuroscience uses richer, more naturalistic tasks — and these tasks are where the framework's explanatory power could truly shine. In this notebook, we design generative models for three canonical cognitive tasks, showing how fundamental phenomena (impulsivity, drift-diffusion, memory decay) emerge naturally from the AIF formalism.

This is where Active Inference meets cognitive neuroscience: not as an abstract theory, but as a concrete modeling framework that makes testable predictions about neural dynamics.

By the end of this notebook you will:
1. Design A/B/C/D matrices for Go/No-Go, evidence accumulation, and working memory tasks
2. See how drift-diffusion dynamics emerge from sequential Bayesian inference
3. Understand working memory as sustained belief under uncertainty
4. Connect AIF belief trajectories to neural recordings
5. Identify the frontier: what scaling these models requires

**Prerequisites:** Modules 8–10 (generative models, EFE), Module 12 (deep AIF).  
**Time:** ~75 minutes.

## A4.1 The Gap: From Toy Problems to Cognitive Tasks

AIF has been demonstrated primarily on:
- Grid worlds (Modules 1–6)
- T-mazes (Module 9)
- Simple POMDPs (Module 10)

But cognitive neuroscience tasks like those in NeuroGym (Appendix A1) are richer. Three tasks, three cognitive constructs:

| Task | Cognitive Construct | Key Phenomenon | AIF Mechanism |
|------|-------------------|----------------|---------------|
| Go/No-Go | Response inhibition | Impulsivity / commission errors | Belief about trial type → action selection |
| Perceptual Decision Making | Evidence accumulation | Drift-diffusion dynamics | Sequential Bayesian updating |
| Delay Match to Sample | Working memory | Memory decay over delay | Belief precision under uninformative observations |

In each case, the key cognitive phenomenon **emerges naturally** from the AIF generative model — no special-purpose mechanisms needed.

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from collections import namedtuple

import sys
sys.path.insert(0, '..')
from utils.plotting import (
    COLORS, _setup_style, dual_perspective_box,
    plot_matrix_heatmap, plot_belief_evolution,
    plot_efe_decomposition, plot_policy_distribution,
)

# Try importing alf framework; provide self-contained fallbacks if unavailable
try:
    from alf.generative_model import GenerativeModel
    from alf.free_energy import expected_free_energy_decomposed, EFEDecomposition
    ALF_AVAILABLE = True
except ImportError:
    ALF_AVAILABLE = False
    # Self-contained fallback implementations below

if not ALF_AVAILABLE:
    EFEDecomposition = namedtuple('EFEDecomposition', ['G_total', 'pragmatic', 'epistemic'])

    def expected_free_energy_decomposed(A, B, C, beliefs, action):
        """Compute EFE decomposed into pragmatic and epistemic terms.

        Parameters
        ----------
        A : ndarray (n_obs, n_states) - likelihood P(o|s)
        B : ndarray (n_states, n_states, n_actions) - transition P(s'|s,a)
        C : ndarray (n_obs,) - log preferences over observations
        beliefs : ndarray (n_states,) - current beliefs over states
        action : int - action index

        Returns
        -------
        EFEDecomposition(G_total, pragmatic, epistemic)
        """
        # Predicted next-state distribution
        predicted_state = B[:, :, action] @ beliefs
        predicted_state = np.clip(predicted_state, 1e-16, None)
        predicted_state /= predicted_state.sum()

        # Predicted observation distribution
        predicted_obs = A @ predicted_state
        predicted_obs = np.clip(predicted_obs, 1e-16, None)
        predicted_obs /= predicted_obs.sum()

        # Pragmatic value: expected log preference
        pragmatic = predicted_obs @ C

        # Epistemic value: negative expected conditional entropy H[o|s]
        H_obs_given_state = -np.sum(A * np.log(A + 1e-16), axis=0)
        epistemic = -predicted_state @ H_obs_given_state

        return EFEDecomposition(
            G_total=pragmatic + epistemic,
            pragmatic=pragmatic,
            epistemic=epistemic,
        )

np.random.seed(42)
_setup_style()

print(f"alf framework available: {ALF_AVAILABLE}")
print("Appendix A4: Deep Active Inference on Cognitive Tasks")
print("=" * 55)

## A4.2 Go/No-Go as Active Inference

The **Go/No-Go task** is a staple of cognitive neuroscience: respond to "go" stimuli, withhold for "no-go" stimuli. Errors of commission (responding to no-go) are clinically relevant markers of **impulsivity** (ADHD, substance use disorders, frontal lesions).

### State Space Design

We model the task as a POMDP with:
- **6 hidden states**: {go, nogo} × {pre_stimulus, stimulus, response_window}
- **5 observations**: ['null', 'weak_stimulus', 'strong_stimulus', 'reward', 'no_reward']
- **3 actions**: ['wait', 'respond', 'withhold']

The key insight: the agent must **infer the trial type** (go vs. no-go) from noisy observations before selecting an action. Commission errors arise when the agent's belief is insufficiently sharp — it responds before it has enough evidence.

In [ ]:
# ── Go/No-Go: Generative Model ────────────────────────────────────────────────
state_names = ['go_pre', 'go_stim', 'go_resp', 'nogo_pre', 'nogo_stim', 'nogo_resp']
obs_names = ['null', 'weak_stim', 'strong_stim', 'reward', 'no_reward']
action_names = ['wait', 'respond', 'withhold']

n_states, n_obs, n_actions = 6, 5, 3

# A matrix: P(observation | hidden state)
A = np.zeros((n_obs, n_states))
# Pre-stimulus: observe null
A[0, 0] = 1.0   # go_pre -> null
A[0, 3] = 1.0   # nogo_pre -> null
# Stimulus phase: go -> strong, nogo -> weak (with overlap)
A[2, 1] = 0.85; A[1, 1] = 0.15   # go_stim -> strong (mostly)
A[1, 4] = 0.85; A[2, 4] = 0.15   # nogo_stim -> weak (mostly)
# Response phase: correct actions -> reward, incorrect -> no_reward
A[3, 2] = 1.0   # go_resp -> reward (responded correctly to go)
A[4, 5] = 1.0   # nogo_resp -> no_reward (responded incorrectly to nogo)

print("A matrix: P(observation | state)")
print(f"Shape: {A.shape}")
print(f"Column sums: {A.sum(axis=0).round(3)}")
print()
for i, obs in enumerate(obs_names):
    for j, state in enumerate(state_names):
        if A[i, j] > 0.01:
            print(f"  P({obs:12s} | {state:10s}) = {A[i,j]:.2f}")

In [ ]:
# ── Go/No-Go: Transition Model B ─────────────────────────────────────────────
# B matrix: P(next_state | current_state, action)
B = np.zeros((n_states, n_states, n_actions))

# Action 0: wait -- advance through task phases
B[1, 0, 0] = 1.0   # go_pre + wait -> go_stim
B[4, 3, 0] = 1.0   # nogo_pre + wait -> nogo_stim
B[1, 1, 0] = 1.0   # go_stim + wait -> go_stim (still waiting)
B[4, 4, 0] = 1.0   # nogo_stim + wait -> nogo_stim (still waiting)
B[2, 2, 0] = 1.0   # go_resp + wait -> go_resp (absorbing)
B[5, 5, 0] = 1.0   # nogo_resp + wait -> nogo_resp (absorbing)

# Action 1: respond
B[1, 0, 1] = 1.0   # go_pre + respond -> go_stim (too early, ignored)
B[4, 3, 1] = 1.0   # nogo_pre + respond -> nogo_stim (too early, ignored)
B[2, 1, 1] = 1.0   # go_stim + respond -> go_resp (CORRECT!)
B[5, 4, 1] = 1.0   # nogo_stim + respond -> nogo_resp (COMMISSION ERROR!)
B[2, 2, 1] = 1.0   # go_resp + respond -> go_resp (absorbing)
B[5, 5, 1] = 1.0   # nogo_resp + respond -> nogo_resp (absorbing)

# Action 2: withhold
B[1, 0, 2] = 1.0   # go_pre + withhold -> go_stim
B[4, 3, 2] = 1.0   # nogo_pre + withhold -> nogo_stim
B[1, 1, 2] = 1.0   # go_stim + withhold -> go_stim (missed go trial!)
B[4, 4, 2] = 1.0   # nogo_stim + withhold -> nogo_stim (CORRECT INHIBITION)
B[2, 2, 2] = 1.0   # go_resp + withhold -> go_resp (absorbing)
B[5, 5, 2] = 1.0   # nogo_resp + withhold -> nogo_resp (absorbing)

# Verify column sums
for a_idx, a_name in enumerate(action_names):
    col_sums = B[:, :, a_idx].sum(axis=0)
    print(f"B[{a_name}] column sums: {col_sums.round(3)}")

In [ ]:
# ── Go/No-Go: Preferences (C) and Prior (D) ──────────────────────────────
# C: log preferences over observations
C = np.zeros(n_obs)
C[3] = 2.0    # prefer reward
C[4] = -2.0   # avoid no_reward

# D: prior over initial states (uniform over pre-stimulus)
D = np.zeros(n_states)
D[0] = 0.5   # P(go_pre) = 0.5
D[3] = 0.5   # P(nogo_pre) = 0.5

print("C (log preferences):")
for i, name in enumerate(obs_names):
    print(f"  C({name:12s}) = {C[i]:+.1f}")

print("\nD (prior over initial states):")
for i, name in enumerate(state_names):
    if D[i] > 0:
        print(f"  P({name}) = {D[i]:.2f}")

In [ ]:
# ── Visualize Go/No-Go Generative Model ──────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# A matrix
plot_matrix_heatmap(A, title='A: P(obs | state)',
                    row_labels=obs_names, col_labels=state_names,
                    cmap='YlOrRd', ax=axes[0])

# B[respond]
plot_matrix_heatmap(B[:, :, 1], title="B[respond]: P(s' | s, respond)",
                    row_labels=state_names, col_labels=state_names,
                    cmap='Blues', ax=axes[1])

# C: preferences
c_colors = [COLORS['reward'] if c > 0 else COLORS['danger'] if c < 0
            else COLORS['neutral'] for c in C]
axes[2].bar(range(n_obs), np.exp(C) / np.exp(C).sum(), color=c_colors)
axes[2].set_xticks(range(n_obs))
axes[2].set_xticklabels(obs_names, rotation=45, ha='right')
axes[2].set_title('C: Preferences (softmax)')
axes[2].set_ylabel('Relative preference')

# D: prior
d_colors = [COLORS['aif'] if d > 0 else COLORS['neutral'] for d in D]
axes[3].bar(range(n_states), D, color=d_colors)
axes[3].set_xticks(range(n_states))
axes[3].set_xticklabels(state_names, rotation=45, ha='right')
axes[3].set_title('D: Prior over initial states')
axes[3].set_ylabel('P(s_0)')

plt.suptitle('Go/No-Go: Complete Generative Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Run Go Trial: Bayesian Belief Updating ─────────────────────────────────
def bayesian_update(prior, A, obs_idx):
    """Single Bayesian belief update: P(s|o) proportional to P(o|s) * P(s)."""
    likelihood = A[obs_idx, :]
    posterior = likelihood * prior
    s = posterior.sum()
    if s < 1e-16:
        return prior.copy()  # fallback: if no mass, keep prior
    return posterior / s


def predict_next_state(beliefs, B, action):
    """Predict next-state beliefs using B matrix: P(s') = B[:,:,a] @ P(s)."""
    next_beliefs = B[:, :, action] @ beliefs
    s = next_beliefs.sum()
    if s < 1e-16:
        return beliefs.copy()
    return next_beliefs / s


# Simulate a GO trial with proper phase transitions
# The trial proceeds: prior -> obs(null) -> transition(wait) -> obs(strong_stim)
beliefs = D.copy()
belief_history = [beliefs.copy()]

# Phase 1: Pre-stimulus -- observe null
beliefs = bayesian_update(beliefs, A, 0)  # obs: null
belief_history.append(beliefs.copy())

# Transition: wait action advances pre -> stim phase via B matrix
beliefs = predict_next_state(beliefs, B, 0)  # action: wait
belief_history.append(beliefs.copy())

# Phase 2: Stimulus -- observe strong_stim (go trial)
beliefs = bayesian_update(beliefs, A, 2)  # obs: strong_stim
belief_history.append(beliefs.copy())

print("Go Trial: Belief evolution")
print("=" * 50)
print(f"\nPrior beliefs:         {D.round(3)}")
print(f"After null obs:        {belief_history[1].round(3)}")
print(f"After wait transition: {belief_history[2].round(3)}")
print(f"After strong_stim:     {belief_history[3].round(3)}")

print("\nAfter strong stimulus:")
for s, b in zip(state_names, beliefs):
    if b > 0.01:
        print(f"  P({s}) = {b:.3f}")

print(f"\nThe agent now strongly believes this is a GO trial in the stimulus phase.")

In [ ]:
# ── Go Trial: Belief Evolution + EFE Decomposition ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: belief evolution
plot_belief_evolution(belief_history, state_names=state_names,
                      title='Go Trial: Belief Evolution', ax=ax1)
ax1.set_xticks([0, 1, 2, 3])
ax1.set_xticklabels(['Prior', 'After null', 'After wait\n(transition)', 'After\nstrong_stim'],
                     fontsize=8)

# Right: EFE decomposition for each action after observing strong stimulus
pragmatic_vals, epistemic_vals = [], []
for a in range(n_actions):
    efe = expected_free_energy_decomposed(A, B, C, beliefs, a)
    pragmatic_vals.append(efe.pragmatic)
    epistemic_vals.append(efe.epistemic)
    print(f"Action '{action_names[a]}': G={efe.G_total:.3f} "
          f"(pragmatic={efe.pragmatic:.3f}, epistemic={efe.epistemic:.3f})")

plot_efe_decomposition(pragmatic_vals, epistemic_vals,
                       action_names=action_names,
                       title='Go Trial: EFE After Strong Stimulus', ax=ax2)

plt.tight_layout()
plt.show()

# Policy from EFE (minimize G, so negate for softmax)
G = np.array([p + e for p, e in zip(pragmatic_vals, epistemic_vals)])
gamma = 4.0
# In AIF convention, we MAXIMIZE G (pragmatic + epistemic), so higher is better
policy = np.exp(gamma * G) / np.exp(gamma * G).sum()
best_action = action_names[np.argmax(policy)]
print(f"\nPolicy (gamma={gamma}): {dict(zip(action_names, policy.round(3)))}")
print(f"Selected action: '{best_action}'")
print("\n'Respond' has highest EFE because it leads to go_resp -> reward (high pragmatic value).")

In [ ]:
# ── Dual Perspective: Go/No-Go ─────────────────────────────────────────────
dual_perspective_box(
    rl_text=(
        "RL learns a direct stimulus-response mapping: strong stimulus -> respond. "
        "This requires many trials and doesn't explain <i>why</i> the agent responds "
        "-- just that past rewards reinforced this action."
    ),
    aif_text=(
        "AIF first <b>infers the trial type</b> (go vs. no-go) from the observation, "
        "then selects the action that minimizes expected free energy. The agent responds "
        "because it <b>believes</b> this is a go trial and <b>prefers</b> reward over "
        "punishment. Commission errors arise from <b>ambiguous beliefs</b>, not noisy "
        "action selection."
    ),
    title="Go/No-Go: Stimulus-Response vs. Belief-Driven Action"
)

## A4.3 Perceptual Decision Making as Active Inference

The **Random Dots Motion task** (Roitman & Shadlen, 2002) is perhaps the most studied perceptual decision paradigm in neuroscience. The subject observes a cloud of dots, some moving coherently left or right, and must decide the direction.

The key insight: **drift-diffusion dynamics ARE Bayesian inference** on a simple generative model. The log-likelihood ratio (left vs. right) is a sufficient statistic that accumulates evidence over time — exactly like the drift-diffusion model (DDM).

### Generative Model

- **2 hidden states**: ['left', 'right']
- **5 observation bins**: discretized noisy evidence from strong_left to strong_right
- **A matrix**: parameterized by motion **coherence** (signal strength)

In [ ]:
# ── Perceptual Decision Making: Generative Model ──────────────────────────
pdm_state_names = ['left', 'right']
pdm_obs_names = ['strong_L', 'weak_L', 'ambiguous', 'weak_R', 'strong_R']


def build_pdm_A(coherence, n_bins=5):
    """Build likelihood matrix for PDM task at given coherence.

    Higher coherence = more informative observations.
    At coherence=0, all observations are equally likely regardless of state.
    """
    A = np.zeros((n_bins, 2))
    # Bin centers from -1 (strong left) to +1 (strong right)
    centers = np.linspace(-1, 1, n_bins)

    for s in range(2):  # 0=left, 1=right
        mu = -1 + 2 * s  # -1 for left, +1 for right
        sigma = 1.0 - 0.5 * coherence + 0.01  # narrower at high coherence
        for o in range(n_bins):
            A[o, s] = np.exp(-0.5 * ((centers[o] - mu * coherence) / sigma) ** 2)

    # Normalize columns
    A /= A.sum(axis=0, keepdims=True)
    return A


# Show A matrices at different coherences
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for idx, coh in enumerate([0.0, 0.2, 0.5, 1.0]):
    A_pdm = build_pdm_A(coh)
    plot_matrix_heatmap(A_pdm, title=f'A (coherence={coh})',
                        row_labels=pdm_obs_names, col_labels=pdm_state_names,
                        cmap='YlOrRd', ax=axes[idx])

plt.suptitle('PDM Likelihood Matrices at Different Coherences', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("At coherence=0: both states produce identical observation distributions (pure noise)")
print("At coherence=1: observations are highly diagnostic of the true state")

In [ ]:
# ── Sequential Bayesian Updating: Drift-Diffusion Emerges ───────────────────
def run_pdm_trial(coherence, true_direction, T=50, threshold=0.95, seed=None):
    """Run one PDM trial with sequential Bayesian updating.

    Parameters
    ----------
    coherence : float in [0, 1]
    true_direction : int, 0=left, 1=right
    T : int, max number of time steps
    threshold : float, decision threshold on P(state)
    seed : int or None

    Returns
    -------
    belief_history : ndarray (n_steps+1, 2)
    rt : int, reaction time (steps to threshold)
    choice : int, chosen direction (argmax of belief)
    """
    rng = np.random.RandomState(seed)
    A_pdm = build_pdm_A(coherence)
    belief = np.array([0.5, 0.5])  # uniform prior
    belief_history = [belief.copy()]

    for t in range(T):
        # Generate noisy observation from true state
        obs_probs = A_pdm[:, true_direction]
        obs = rng.choice(len(obs_probs), p=obs_probs)

        # Bayesian update
        likelihood = A_pdm[obs, :]
        belief = likelihood * belief
        belief /= belief.sum()
        belief_history.append(belief.copy())

        # Check threshold (decision)
        if np.max(belief) > threshold:
            return np.array(belief_history), t + 1, np.argmax(belief)

    return np.array(belief_history), T, np.argmax(belief)


# Run a single trial at moderate coherence
bh, rt, choice = run_pdm_trial(0.3, true_direction=1, seed=42)
print(f"Coherence=0.3, true=right")
print(f"  Decision: {'right' if choice == 1 else 'left'} (correct={choice == 1})")
print(f"  Reaction time: {rt} steps")
print(f"  Final belief: P(left)={bh[-1, 0]:.3f}, P(right)={bh[-1, 1]:.3f}")

In [ ]:
# ── Plot Single Trial: Drift-Diffusion Pattern ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (coh, seed) in enumerate([(0.1, 10), (0.3, 42), (0.8, 7)]):
    bh, rt, choice = run_pdm_trial(coh, true_direction=1, seed=seed)

    # Plot P(right) - 0.5: centers the drift at zero
    log_ratio = np.log(bh[:, 1] / (bh[:, 0] + 1e-16) + 1e-16)

    axes[idx].plot(range(len(bh)), bh[:, 1], color=COLORS['aif'], linewidth=2,
                   label='P(right)')
    axes[idx].plot(range(len(bh)), bh[:, 0], color=COLORS['rl'], linewidth=2,
                   label='P(left)')
    axes[idx].axhline(y=0.95, color=COLORS['reward'], linestyle='--', alpha=0.5,
                      label='Threshold')
    axes[idx].axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)

    # Mark decision point
    if rt < 50:
        axes[idx].axvline(x=rt, color=COLORS['highlight'], linestyle='--', alpha=0.7)
        axes[idx].text(rt + 0.5, 0.55, f'RT={rt}', fontsize=10, color=COLORS['highlight'])

    correct = (choice == 1)
    axes[idx].set_title(f'Coherence={coh} | {"Correct" if correct else "Error"} | RT={rt}')
    axes[idx].set_xlabel('Time step')
    axes[idx].set_ylabel('Belief')
    axes[idx].set_ylim(-0.05, 1.05)
    axes[idx].legend(fontsize=9)

plt.suptitle('Perceptual Decision Making: Belief Trajectories (true direction = right)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice the drift-diffusion pattern: beliefs accumulate evidence stochastically.")
print("Higher coherence produces faster, more reliable drift toward the correct state.")

In [ ]:
# ── Multiple Trials at Different Coherences ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
n_trials_plot = 10

for idx, coh in enumerate([0.05, 0.3, 0.8]):
    for trial in range(n_trials_plot):
        bh, rt, choice = run_pdm_trial(coh, true_direction=1, seed=trial * 7 + idx)
        correct = (choice == 1)
        color = COLORS['reward'] if correct else COLORS['danger']
        axes[idx].plot(range(len(bh)), bh[:, 1], color=color, alpha=0.4, linewidth=1)

    axes[idx].axhline(y=0.95, color='black', linestyle='--', alpha=0.3, label='Threshold')
    axes[idx].axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.3)
    axes[idx].set_title(f'Coherence = {coh}')
    axes[idx].set_xlabel('Time step')
    axes[idx].set_ylabel('P(right)')
    axes[idx].set_ylim(-0.05, 1.05)
    axes[idx].legend(fontsize=9)

# Add custom legend
import matplotlib.patches as mpatches
correct_patch = mpatches.Patch(color=COLORS['reward'], alpha=0.6, label='Correct')
error_patch = mpatches.Patch(color=COLORS['danger'], alpha=0.6, label='Error')
axes[2].legend(handles=[correct_patch, error_patch], fontsize=9)

plt.suptitle('Multiple Trials: Belief Trajectories (green=correct, red=error)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Low coherence: slow, noisy drift with many errors")
print("High coherence: fast, clean drift to correct answer")

In [ ]:
# ── Psychometric & Chronometric Curves ────────────────────────────────────
coherences = [0.0, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
accuracies, mean_rts, std_rts = [], [], []
n_sim = 200

for coh in coherences:
    correct, rts = 0, []
    for trial_i in range(n_sim):
        _, rt, choice = run_pdm_trial(coh, true_direction=1, seed=trial_i)
        correct += (choice == 1)
        rts.append(rt)
    accuracies.append(correct / n_sim)
    mean_rts.append(np.mean(rts))
    std_rts.append(np.std(rts))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Psychometric curve: accuracy vs coherence
ax1.plot(coherences, accuracies, 'o-', color=COLORS['aif'], linewidth=2, markersize=8)
ax1.axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5, label='Chance')
ax1.set_xlabel('Motion Coherence')
ax1.set_ylabel('Accuracy (proportion correct)')
ax1.set_title('Psychometric Curve')
ax1.set_ylim(0.4, 1.05)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Chronometric curve: RT vs coherence
ax2.errorbar(coherences, mean_rts, yerr=std_rts, fmt='o-', color=COLORS['rl'],
             linewidth=2, markersize=8, capsize=4)
ax2.set_xlabel('Motion Coherence')
ax2.set_ylabel('Mean Reaction Time (steps)')
ax2.set_title('Chronometric Curve')
ax2.grid(True, alpha=0.3)

plt.suptitle('Bayesian PDM Reproduces Classic Behavioral Signatures',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Psychometric: accuracy increases sigmoidally with coherence")
print("Chronometric: RT decreases with coherence (easier = faster)")
print("Both signatures match empirical data from Roitman & Shadlen (2002).")

### Key Insight: DDM IS Variational Inference

The **drift-diffusion model** (DDM) is the standard model for perceptual decision making in neuroscience (Ratcliff, 1978). It describes a noisy evidence accumulation process with a constant drift rate and absorbing decision boundaries.

Our Bayesian model produces identical dynamics because the **log-likelihood ratio** is a sufficient statistic:

$$\text{LLR}_t = \log \frac{P(s=\text{right} | o_{1:t})}{P(s=\text{left} | o_{1:t})} = \sum_{\tau=1}^{t} \log \frac{P(o_\tau | s=\text{right})}{P(o_\tau | s=\text{left})}$$

Each observation adds an independent increment to the LLR. This sum of i.i.d. random variables produces a random walk with drift — which IS the drift-diffusion process.

| DDM Parameter | Bayesian Equivalent |
|---|---|
| Drift rate | Expected log-likelihood ratio per observation |
| Diffusion noise | Variance of the log-likelihood ratio |
| Decision boundary | Belief threshold (e.g., P > 0.95) |
| Starting point bias | Prior $P(s)$ |
| Non-decision time | Pre-evidence accumulation latency |

Active Inference adds something the DDM lacks: an explanation of **why** the agent accumulates evidence (to minimize expected free energy) and **when** to stop (when further evidence has negligible expected value).

## A4.4 Working Memory as Sustained Belief

The **Delay Match to Sample** (DMS) task (Fuster & Alexander, 1971) is the canonical working memory paradigm:

1. **Sample phase**: A stimulus is presented briefly
2. **Delay phase**: The stimulus is removed; the subject must remember it
3. **Test phase**: A probe stimulus appears; the subject reports if it matches

In AIF terms, working memory is **maintaining beliefs about hidden states when observations are uninformative**. During the delay, the observation likelihood becomes flat (every state equally likely to produce "nothing"), so beliefs must be sustained by their own precision against the pull of the prior.

This is where the **precision** parameter becomes critical: it controls how tightly beliefs are maintained.

In [ ]:
# ── Working Memory: Generative Model ───────────────────────────────────────
wm_state_names = ['stimulus_A', 'stimulus_B']
wm_obs_names = ['see_A', 'see_B', 'nothing']

# A matrices for different phases
# Sample phase: highly informative
A_sample = np.array([
    [0.95, 0.05],   # see_A | stimulus_A vs stimulus_B
    [0.05, 0.95],   # see_B
    [0.00, 0.00],   # nothing (never happens during sample)
])
# Fix: ensure columns sum to 1
A_sample[2, :] = 1.0 - A_sample[:2, :].sum(axis=0)

# Delay phase: completely uninformative
A_delay = np.array([
    [0.01, 0.01],   # see_A (very rare)
    [0.01, 0.01],   # see_B (very rare)
    [0.98, 0.98],   # nothing (dominant)
])

# Test phase: informative again
A_test = A_sample.copy()

print("A matrices for each phase:")
print(f"\nSample (informative):")
for o_idx, obs in enumerate(wm_obs_names):
    print(f"  P({obs:8s} | A) = {A_sample[o_idx, 0]:.2f}, P({obs:8s} | B) = {A_sample[o_idx, 1]:.2f}")

print(f"\nDelay (uninformative):")
for o_idx, obs in enumerate(wm_obs_names):
    print(f"  P({obs:8s} | A) = {A_delay[o_idx, 0]:.2f}, P({obs:8s} | B) = {A_delay[o_idx, 1]:.2f}")

# Verify columns sum to 1
print(f"\nColumn sums - Sample: {A_sample.sum(axis=0).round(3)}")
print(f"Column sums - Delay:  {A_delay.sum(axis=0).round(3)}")

In [ ]:
# ── Run DMS Trial: Sample -> Delay -> Test ────────────────────────────────
def run_dms_trial(true_stimulus, delay_length, prior_precision=1.0, seed=42):
    """Run a Delay Match to Sample trial.

    Parameters
    ----------
    true_stimulus : int, 0=A, 1=B
    delay_length : int, number of delay time steps
    prior_precision : float, how strongly the prior pulls beliefs toward uniform
    seed : int

    Returns
    -------
    belief_history : list of arrays
    phase_labels : list of str
    """
    rng = np.random.RandomState(seed)

    # Prior: slightly biased toward uniform (no preference)
    prior = np.array([0.5, 0.5])
    beliefs = prior.copy()
    belief_history = [beliefs.copy()]
    phase_labels = ['prior']

    # --- Sample phase (3 time steps) ---
    for t in range(3):
        obs_probs = A_sample[:, true_stimulus]
        obs = rng.choice(3, p=obs_probs)
        beliefs = bayesian_update(beliefs, A_sample, obs)
        belief_history.append(beliefs.copy())
        phase_labels.append('sample')

    # --- Delay phase ---
    for t in range(delay_length):
        obs_probs = A_delay[:, true_stimulus]
        obs = rng.choice(3, p=obs_probs)

        # Bayesian update with uninformative observations
        beliefs = bayesian_update(beliefs, A_delay, obs)

        # Prior precision pull: blend toward prior (models memory decay)
        decay_rate = 0.02 * (1.0 / prior_precision)
        beliefs = (1 - decay_rate) * beliefs + decay_rate * prior
        beliefs /= beliefs.sum()

        belief_history.append(beliefs.copy())
        phase_labels.append('delay')

    # --- Test phase (3 time steps) ---
    for t in range(3):
        obs_probs = A_test[:, true_stimulus]
        obs = rng.choice(3, p=obs_probs)
        beliefs = bayesian_update(beliefs, A_test, obs)
        belief_history.append(beliefs.copy())
        phase_labels.append('test')

    return belief_history, phase_labels


# Run a trial with stimulus A and moderate delay
bh, phases = run_dms_trial(true_stimulus=0, delay_length=15, prior_precision=1.0)

print(f"Trial: stimulus=A, delay=15 steps")
print(f"  After sample:  P(A) = {bh[3][0]:.3f}")
print(f"  End of delay:  P(A) = {bh[3+15][0]:.3f}")
print(f"  After test:    P(A) = {bh[-1][0]:.3f}")

In [ ]:
# ── Visualize DMS Belief Dynamics ──────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(14, 5))

beliefs_arr = np.array(bh)
T_total = len(beliefs_arr)

# Plot P(stimulus_A) over time
ax.plot(range(T_total), beliefs_arr[:, 0], color=COLORS['aif'], linewidth=2.5,
        label='P(stimulus_A)')
ax.plot(range(T_total), beliefs_arr[:, 1], color=COLORS['rl'], linewidth=2.5,
        label='P(stimulus_B)')
ax.axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)

# Shade phases
sample_end = phases.index('delay') if 'delay' in phases else len(phases)
delay_end = len(phases) - 1 - phases[::-1].index('delay') if 'delay' in phases else sample_end

ax.axvspan(0, sample_end, alpha=0.1, color=COLORS['reward'], label='Sample phase')
ax.axvspan(sample_end, delay_end + 1, alpha=0.1, color=COLORS['neutral'], label='Delay phase')
ax.axvspan(delay_end + 1, T_total, alpha=0.1, color=COLORS['epistemic'], label='Test phase')

ax.set_xlabel('Time step')
ax.set_ylabel('Belief')
ax.set_title('Working Memory: Belief Dynamics in Delay Match to Sample (true stimulus = A)')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='center right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("During SAMPLE: beliefs sharpen quickly (informative observations)")
print("During DELAY: beliefs slowly decay toward the prior (uninformative observations)")
print("During TEST: beliefs re-sharpen (informative observations return)")

In [ ]:
# ── Compare Short vs Long Delays ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
delay_lengths = [5, 20, 50]

for idx, delay in enumerate(delay_lengths):
    bh_d, phases_d = run_dms_trial(true_stimulus=0, delay_length=delay, prior_precision=1.0)
    beliefs_d = np.array(bh_d)

    axes[idx].plot(range(len(beliefs_d)), beliefs_d[:, 0], color=COLORS['aif'],
                   linewidth=2.5, label='P(A)')
    axes[idx].plot(range(len(beliefs_d)), beliefs_d[:, 1], color=COLORS['rl'],
                   linewidth=2.5, label='P(B)')
    axes[idx].axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)

    # Shade delay
    sample_t = 4  # prior + 3 sample steps
    delay_end_t = sample_t + delay
    axes[idx].axvspan(sample_t, delay_end_t, alpha=0.15, color=COLORS['neutral'])

    end_delay_belief = beliefs_d[delay_end_t, 0]
    axes[idx].set_title(f'Delay = {delay} steps\nP(A) at delay end: {end_delay_belief:.3f}')
    axes[idx].set_xlabel('Time step')
    axes[idx].set_ylabel('Belief')
    axes[idx].set_ylim(-0.05, 1.05)
    axes[idx].legend(fontsize=9)

plt.suptitle('Working Memory Decay: Longer Delays Degrade Belief Precision',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Short delay: beliefs remain sharp through the delay period")
print("Long delay: beliefs decay significantly, approaching the prior (0.5)")
print("This matches the well-known finding that WM accuracy degrades with delay.")

In [ ]:
# ── Effect of Prior Precision on Memory Maintenance ───────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

precisions = [0.5, 1.0, 2.0, 5.0]
delay_test = 30
colors_prec = [COLORS['danger'], COLORS['highlight'], COLORS['aif'], COLORS['epistemic']]

for prec, color in zip(precisions, colors_prec):
    bh_p, _ = run_dms_trial(true_stimulus=0, delay_length=delay_test,
                            prior_precision=prec, seed=42)
    beliefs_p = np.array(bh_p)
    ax.plot(range(len(beliefs_p)), beliefs_p[:, 0], color=color,
            linewidth=2, label=f'Precision = {prec}')

# Shade delay
ax.axvspan(4, 4 + delay_test, alpha=0.1, color=COLORS['neutral'])
ax.axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)

ax.set_xlabel('Time step')
ax.set_ylabel('P(stimulus_A)')
ax.set_title('Effect of Prior Precision on Working Memory Maintenance')
ax.set_ylim(0.3, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Annotate
ax.text(4 + delay_test / 2, 0.35, 'Delay Period', ha='center', fontsize=11,
        color=COLORS['neutral'], fontstyle='italic')

plt.tight_layout()
plt.show()

print("Higher precision = stronger memory maintenance (less decay during delay)")
print("Lower precision = faster decay toward the uninformative prior")
print("\nIn AIF, precision maps to attention / cognitive control:")
print("  - High precision: strong PFC sustained activity (focused attention)")
print("  - Low precision: distractible, poor WM (ADHD-like pattern)")

### Connection to Precision and Attention in Active Inference

In the full AIF framework, **precision** ($\gamma$, $\beta$, or $\omega$) modulates the weight of different information sources. For working memory:

- **High belief precision during delay** = strong WM maintenance = PFC sustained firing
- **Low belief precision** = poor WM = beliefs decay rapidly = distractibility
- **Attention** corresponds to optimizing precision: the brain allocates computational resources to maintain task-relevant beliefs

This connects directly to the **precision-weighting** account of attention in predictive processing (Feldman & Friston, 2010): attention IS precision optimization.

| WM Phenomenon | AIF Mechanism |
|---|---|
| Memory maintenance | High precision on posterior beliefs |
| Memory decay | Prior pulls beliefs toward uniform |
| Distraction | Unexpected observations reduce posterior precision |
| Capacity limits | Finite precision budget across state dimensions |
| Rehearsal | Active re-generation of informative "virtual" observations |

## A4.5 From Discrete to Deep: Scaling Up

The discrete models above are **interpretable** but require hand-designed state spaces. For real neuroscience applications (continuous observations, high-dimensional stimuli, multi-step tasks), we need **deep Active Inference** (Module 12).

The conceptual mapping from discrete to deep:

| Discrete Component | Deep Replacement | Advantage |
|---|---|---|
| A matrix (tabular) | Encoder network $P(s\|o)$ | Scales to images, continuous obs |
| B matrix (tabular) | Transition network $P(s'\|s,a)$ | Handles continuous dynamics |
| C vector (fixed) | Learned preference model | Adapts to context |
| D vector (fixed) | Learned prior | Captures complex initial conditions |
| Exact Bayesian update | Amortized inference | Constant-time inference |

In [ ]:
# ── Discrete vs Deep: Comparison for Each Task ────────────────────────────
comparison = [
    ['Task', 'Discrete (this notebook)', 'Deep AIF (DR-FREE / Neural)'],
    ['Go/No-Go',
     '6 states, 5 obs, hand-designed A/B',
     'Continuous obs from NeuroGym, encoder learns stimulus features'],
    ['PDM',
     '2 states, 5 bins, analytical update',
     'Raw pixel motion, learned evidence accumulation'],
    ['DMS',
     '2 states, 3 obs, manual phase switching',
     'Image stimuli, temporal dynamics learned end-to-end'],
    ['Scaling',
     'O(n_states^2 * n_actions) per step',
     'O(1) amortized via neural net forward pass'],
]

# Print table
print(f"{'Task':<16} {'Discrete (this notebook)':<45} {'Deep AIF (DR-FREE / Neural)'}")
print("─" * 110)
for row in comparison[1:]:
    print(f"{row[0]:<16} {row[1]:<45} {row[2]}")

print("\n" + "=" * 110)
print("Key reference: Champion et al. (2025) DR-FREE, Nature Communications")
print("DR-FREE trains deep AIF agents that generalize under distribution shift --")
print("exactly the robustness needed for real cognitive task modeling.")

## A4.6 Neural Data Connection

A major selling point of AIF for cognitive neuroscience: belief trajectories are directly comparable to neural firing rates.

- **LIP neurons** in macaque visual cortex ramp during perceptual decision making (Roitman & Shadlen, 2002) — just like our PDM belief trajectories
- **PFC neurons** sustain elevated firing during working memory delays (Fuster & Alexander, 1971) — just like our DMS belief maintenance
- **ACC neurons** signal surprise and prediction errors — just like AIF free energy

This is not a coincidence. If the brain implements approximate Bayesian inference, then **neural firing rates should track posterior beliefs**, and their dynamics should match the AIF predictions.

In [ ]:
# ── Neural Data Comparison: AIF Beliefs vs Schematic Neural Activity ───────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Top Left: PDM belief trajectory ---
bh_pdm, rt_pdm, _ = run_pdm_trial(0.4, true_direction=1, T=40, seed=42)
axes[0, 0].plot(range(len(bh_pdm)), bh_pdm[:, 1], color=COLORS['aif'], linewidth=2.5)
axes[0, 0].axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)
axes[0, 0].axhline(y=0.95, color=COLORS['reward'], linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Time step')
axes[0, 0].set_ylabel('P(rightward motion)')
axes[0, 0].set_title('AIF Model: Belief Accumulation (PDM)')
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].grid(True, alpha=0.3)

# --- Top Right: Schematic LIP neuron ---
# Simulate a schematic LIP ramping neuron (inspired by Roitman & Shadlen 2002)
np.random.seed(7)
t_neural = np.arange(0, 40)
baseline = 20  # spikes/s
ramp_rate = 1.2  # spikes/s per time step
noise = np.random.normal(0, 3, len(t_neural))
lip_activity = baseline + ramp_rate * t_neural + noise
lip_activity = np.clip(lip_activity, 5, 70)

axes[0, 1].plot(t_neural, lip_activity, color=COLORS['epistemic'], linewidth=2.5)
axes[0, 1].fill_between(t_neural, lip_activity - 3, lip_activity + 3,
                         color=COLORS['epistemic'], alpha=0.15)
axes[0, 1].axhline(y=baseline, color=COLORS['neutral'], linestyle=':', alpha=0.5)
axes[0, 1].axhline(y=60, color=COLORS['reward'], linestyle='--', alpha=0.5,
                    label='Decision threshold')
axes[0, 1].set_xlabel('Time (ms, schematic)')
axes[0, 1].set_ylabel('Firing rate (spikes/s)')
axes[0, 1].set_title('Schematic LIP Neuron (after Roitman & Shadlen 2002)')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# --- Bottom Left: DMS belief trajectory ---
bh_dms, _ = run_dms_trial(true_stimulus=0, delay_length=20, prior_precision=1.5)
beliefs_dms = np.array(bh_dms)
axes[1, 0].plot(range(len(beliefs_dms)), beliefs_dms[:, 0], color=COLORS['aif'], linewidth=2.5)
axes[1, 0].axhline(y=0.5, color=COLORS['neutral'], linestyle=':', alpha=0.5)
axes[1, 0].axvspan(4, 24, alpha=0.1, color=COLORS['neutral'])
axes[1, 0].set_xlabel('Time step')
axes[1, 0].set_ylabel('P(stimulus_A)')
axes[1, 0].set_title('AIF Model: Belief Maintenance (DMS)')
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].grid(True, alpha=0.3)

# --- Bottom Right: Schematic PFC neuron ---
np.random.seed(12)
t_pfc = np.arange(0, 27)
pfc_activity = np.zeros(27)
# Sample phase (0-3): ramp up
pfc_activity[:4] = [10, 25, 35, 40]
# Delay phase (4-23): sustained with slow decay + noise
for t in range(4, 24):
    pfc_activity[t] = 38 - 0.3 * (t - 4) + np.random.normal(0, 2)
# Test phase (24-26): re-activation
pfc_activity[24:] = [35, 42, 45]

axes[1, 1].plot(t_pfc, pfc_activity, color=COLORS['epistemic'], linewidth=2.5)
axes[1, 1].fill_between(t_pfc, pfc_activity - 2, pfc_activity + 2,
                         color=COLORS['epistemic'], alpha=0.15)
axes[1, 1].axhline(y=10, color=COLORS['neutral'], linestyle=':', alpha=0.5,
                    label='Baseline')
axes[1, 1].axvspan(4, 24, alpha=0.1, color=COLORS['neutral'])
axes[1, 1].set_xlabel('Time (schematic)')
axes[1, 1].set_ylabel('Firing rate (spikes/s)')
axes[1, 1].set_title('Schematic PFC Neuron (after Fuster & Alexander 1971)')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('AIF Belief Trajectories Mirror Neural Dynamics',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Left column: AIF model predictions (belief trajectories)")
print("Right column: Schematic neural data (inspired by classic recordings)")
print("\nThe qualitative match is striking:")
print("  - LIP ramps ~ belief accumulation in PDM")
print("  - PFC sustained activity ~ belief maintenance in DMS")

## A4.7 Exercises

### Exercise 1 (Guided): A-Matrix Precision in PDM

Modify the `build_pdm_A` function to accept a `noise_scale` parameter that controls observation precision independently of coherence. What happens to accuracy and RT when you:
- Increase noise (wider Gaussians in A)?
- Decrease noise (sharper Gaussians)?

This manipulates the quality of sensory evidence, analogous to degraded visual conditions.

In [ ]:
# ── Exercise 1: A-Matrix Precision in PDM ─────────────────────────────────
def build_pdm_A_noisy(coherence, noise_scale=1.0, n_bins=5):
    """Like build_pdm_A but with adjustable noise_scale.

    noise_scale < 1: sharper (more precise) observations
    noise_scale > 1: noisier (less precise) observations
    """
    A = np.zeros((n_bins, 2))
    centers = np.linspace(-1, 1, n_bins)
    for s in range(2):
        mu = -1 + 2 * s
        sigma = (1.0 - 0.5 * coherence + 0.01) * noise_scale
        for o in range(n_bins):
            A[o, s] = np.exp(-0.5 * ((centers[o] - mu * coherence) / sigma) ** 2)
    A /= A.sum(axis=0, keepdims=True)
    return A


# Compare different noise levels at coherence=0.3
noise_levels = [0.5, 1.0, 2.0]
n_test = 200
coh_test = 0.3

print(f"PDM at coherence={coh_test}, {n_test} trials per condition")
print(f"{'Noise Scale':<15} {'Accuracy':<12} {'Mean RT':<12}")
print("─" * 40)

for ns in noise_levels:
    A_noisy = build_pdm_A_noisy(coh_test, noise_scale=ns)
    correct, rts_ex = 0, []
    for i in range(n_test):
        rng_ex = np.random.RandomState(i + 100)
        belief_ex = np.array([0.5, 0.5])
        for t in range(50):
            obs_p = A_noisy[:, 1]
            obs_ex = rng_ex.choice(5, p=obs_p)
            belief_ex = A_noisy[obs_ex, :] * belief_ex
            belief_ex /= belief_ex.sum()
            if np.max(belief_ex) > 0.95:
                break
        correct += (np.argmax(belief_ex) == 1)
        rts_ex.append(t + 1)

    print(f"{ns:<15.1f} {correct/n_test:<12.3f} {np.mean(rts_ex):<12.1f}")

print("\nTry it: modify noise_scale and coherence to explore the accuracy-speed tradeoff.")

### Exercise 2 (Intermediate): Extend DMS to 4 Stimuli

The current DMS model uses 2 stimuli. Extend it to 4:
- **4 hidden states**: ['stim_A', 'stim_B', 'stim_C', 'stim_D']
- **5 observations**: ['see_A', 'see_B', 'see_C', 'see_D', 'nothing']

Questions to answer:
1. Does memory decay faster with more items? (Hint: the prior is now 0.25 per item, not 0.5)
2. How does the delay-accuracy curve change with 4 vs 2 stimuli?
3. What precision level is needed to maintain 4-item WM for 20 steps?

In [ ]:
# ── Exercise 2: 4-Stimulus DMS (Starter Code) ────────────────────────────
# TODO: Define A_sample_4 (5x4 matrix), A_delay_4 (5x4 matrix)
# TODO: Modify run_dms_trial to accept A matrices as parameters
# TODO: Compare 2-stimulus vs 4-stimulus memory decay curves

n_stim_4 = 4
wm4_state_names = ['stim_A', 'stim_B', 'stim_C', 'stim_D']
wm4_obs_names = ['see_A', 'see_B', 'see_C', 'see_D', 'nothing']

# Sample A-matrix: each stimulus produces its observation with high probability
A_sample_4 = np.zeros((5, 4))
for i in range(4):
    A_sample_4[i, i] = 0.90  # correct observation
    for j in range(4):
        if j != i:
            A_sample_4[j, i] = 0.025  # confusion
    A_sample_4[4, i] = 1.0 - A_sample_4[:4, i].sum()  # nothing

print("4-Stimulus A_sample:")
print(A_sample_4.round(3))
print(f"Column sums: {A_sample_4.sum(axis=0).round(3)}")
print("\nExtend this by implementing the full 4-stimulus DMS trial.")
print("Compare how quickly beliefs decay toward P=0.25 vs P=0.5 in the 2-stimulus case.")

### Exercise 3 (Open-ended): Design a ContextDecisionMaking Model

The **ContextDecisionMaking** task in NeuroGym (Appendix A1) requires attending to one of two stimulus modalities based on a context cue. Design a generative model for this task:

1. **Define the state space**: What hidden states capture the task structure? Consider: context (modality 1 vs 2), stimulus identity, and task phase.
2. **Design the A matrix**: How do observations depend on the combination of context and stimulus?
3. **Design the B matrix**: How do actions (choose left vs right) interact with context?
4. **Set C**: What does the agent prefer? (Correct responses based on the attended modality.)

This is a genuine research-level exercise. A good solution would be publishable as part of a modeling study.

---

## Further Reading

- **Friston, K.J., FitzGerald, T., Rigoli, F., Schwartenbeck, P., & Pezzulo, G. (2017).** Active Inference: A Process Theory. *Neural Computation*, 29(1). -- The process theory connecting AIF to neural dynamics.
- **Gold, J.I. & Shadlen, M.N. (2007).** The Neural Basis of Decision Making. *Annual Review of Neuroscience*, 30. -- Comprehensive review of evidence accumulation in neural circuits.
- **Fuster, J.M. & Alexander, G.E. (1971).** Neuron Activity Related to Short-Term Memory. *Science*, 173. -- The classic PFC sustained activity finding.
- **Champion, T. et al. (2025).** DR-FREE: Distributionally Robust Free Energy Estimation. *Nature Communications*. -- Deep AIF agents robust to distribution shift.
- **Roitman, J.D. & Shadlen, M.N. (2002).** Response of Neurons in the Lateral Intraparietal Area during a Combined Visual Discrimination Reaction Time Task. *Journal of Neuroscience*, 22(21). -- LIP neuron ramping during PDM.
- **Ratcliff, R. (1978).** A Theory of Memory Retrieval. *Psychological Review*, 85(2). -- The drift-diffusion model.
- **Feldman, H. & Friston, K.J. (2010).** Attention, Uncertainty, and Free-Energy. *Frontiers in Human Neuroscience*, 4. -- Precision-weighting account of attention.
- **Smith, R., Friston, K.J. & Whyte, C.J. (2022).** A Step-by-Step Tutorial on Active Inference. *Journal of Mathematical Psychology*, 107. -- Clearest tutorial on AIF.

---

*Appendix A4 | Spinning Up in Active Inference*